# Data Science for Business - Explaining House Prices

*Session 3 · ISLP 3.1–3.3 (3.3.1 Qualitative Predictors, 3.3.2 Extensions, 3.3.3 Potential Problems) · 5.1.1 · Lab 3.6*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
import statsmodels.formula.api as smf

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

### Real estate appraisal

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to explain the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_03/ameshousing.csv')

## Prepare data

Perform a random 80:20 train/test split. This is the *validation set approach* (ISLP 5.1.1). We split first and explore only the training data, so nothing we learn is influenced by the test set.

In [ ]:
data_train, data_test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
data_train.head()

In [ ]:
data_train.shape

In [ ]:
data_train.columns

Let us first focus on some easy to understand variables.

In [ ]:
data_train = data_train[["SalePrice", "Neighborhood", "HouseStyle", "LotArea", "GrLivArea", "FullBath", "BedroomAbvGr", "KitchenAbvGr", "OverallQual", "OverallCond"]]

In [ ]:
data_train.head()

## Exploratory data analysis

### Descriptive summary statistics

We can quickly calculate the most important summary statistics for a variable with *describe()*.

In [ ]:
data_train["SalePrice"].describe()

### Visualize distribution of single variables

In the following, we will use histograms and density plots to get a feeling of the distribution of our main variables. See https://seaborn.pydata.org/generated/seaborn.displot.html for more information.

Let's first look at the response (*SalePrice*).

In [ ]:
sns.displot(data_train, x="SalePrice", kde=True)
plt.show()

Let's look at some numerical predictors. We will start with *GrLivArea* (above-ground living area in sq ft).

In [ ]:
sns.displot(data_train, x="GrLivArea", kde=True)
plt.show()

**Your turn!** Plot the distribution of the variable *OverallQual*.

In [ ]:
# YOUR CODE HERE

We can also visualize the distribution of categorical variables using *catplot()*. See https://seaborn.pydata.org/generated/seaborn.catplot.html for more information.

Let's start with *Neighborhood*.

In [ ]:
sns.catplot(data_train, x="Neighborhood", kind="count")
plt.xticks(rotation=90)
plt.show()

**Your turn!** Do the same for *HouseStyle*.

In [ ]:
# YOUR CODE HERE

### Relationship between the response and numerical predictors

In the following, we will use scatter plots to visually explore the relationship between *SalePrice* (y-axis) and numerical predictors (x-axis). See https://seaborn.pydata.org/generated/seaborn.relplot.html for more information.

*SalePrice* and *LotArea*.

In [ ]:
sns.relplot(data=data_train, x="LotArea", y="SalePrice", alpha=0.3)
plt.show()

The function *regplot()* is an alternative to *relplot()* that also draws a linear regression fit. See https://seaborn.pydata.org/generated/seaborn.regplot.html for more information.

In [ ]:
sns.regplot(data=data_train, x="LotArea", y="SalePrice", scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
plt.show()

### Relationship between the response and categorical predictors

In the following, we will use box plots to visually explore the relationship between *SalePrice* (y-axis) and categorical predictors (x-axis).

*SalePrice* and *OverallQual*.

In [ ]:
sns.catplot(data=data_train, x="OverallQual", y="SalePrice", kind="box")
plt.show()

**Your task:** Create a box plot of *SalePrice* by *OverallCond*. Is the relationship as clear as for *OverallQual*?

In [ ]:
# YOUR CODE HERE

Violin plots are similar to box plots, except that they also show the probability density of the data at different values.

In [ ]:
sns.catplot(data=data_train, x="OverallCond", y="SalePrice", kind="violin", height=5, aspect=2)
plt.xticks(rotation=90)
plt.show()

## Train linear models

After getting a feeling for the data, we are now ready to fit some linear regression models. We will use the *statsmodels* package, esp. the formula API, which lets us specify R-style formulas. See https://www.statsmodels.org/stable/example_formulas.html for more information.

The ISLP lab (3.6) builds the model matrix with `ModelSpec` and fits it with `sm.OLS`. The formula API gives identical estimates; only the syntax differs.

### Simple linear regression

Let's start with a simple model that includes only one predictor, *GrLivArea* (ISLP 3.1).

In [ ]:
mod_01 = smf.ols(formula='SalePrice ~ GrLivArea', data=data_train)
mod_01 = mod_01.fit()
print(mod_01.summary())

The `GrLivArea` coefficient is the expected change in SalePrice ($) per extra sq ft. The p-value tests $H_0:\beta_1=0$; $R^2$ is the share of variance in SalePrice explained by the model (ISLP 3.1.2–3.1.3).

**Your turn!** Create a simple linear regression model with *BedroomAbvGr* as the only predictor. Note the sign of its coefficient.

In [ ]:
# YOUR CODE HERE

### Multiple linear regression

Let's create a multiple linear regression model with both *GrLivArea* and *BedroomAbvGr* as predictors (ISLP 3.2).

In [ ]:
mod_02 = smf.ols(formula='SalePrice ~ GrLivArea + BedroomAbvGr', data=data_train)
mod_02 = mod_02.fit()
print(mod_02.summary2())

Compare the *BedroomAbvGr* coefficient with your simple model. Holding living area fixed, an extra bedroom means smaller rooms, so the sign can flip (ISLP 3.2).

### Qualitative predictors

Fit a linear regression model with a categorical predictor (*HouseStyle*).

In [ ]:
mod_cat = smf.ols(formula='SalePrice ~ HouseStyle', data=data_train)
mod_cat = mod_cat.fit()
print(mod_cat.summary2())

Statsmodels creates a dummy variable per HouseStyle except the *baseline*; each coefficient is the difference in mean SalePrice to the baseline (ISLP 3.3.1). The formula API picks the baseline alphabetically, here `1.5Fin`, which is not in the table; its mean is the intercept.

In [ ]:
data_train["HouseStyle"].value_counts()

### Interaction effects

Fit a linear regression model with an interaction term between two numerical predictors. `GrLivArea * LotArea` expands to both main effects plus their product `GrLivArea:LotArea`, so the effect of living area can depend on lot size (ISLP 3.3.2).

In [ ]:
mod_int_num = smf.ols(formula='SalePrice ~ GrLivArea * LotArea', data=data_train)
mod_int_num = mod_int_num.fit()
print(mod_int_num.summary2())

Fit a linear regression model with an interaction term between a categorical and a numerical predictor. In the formula API, the *C()* function can be used to indicate that a numerical variable should be treated as categorical. Each quality level now gets its own slope for *GrLivArea*.

In [ ]:
mod_int_cat = smf.ols(formula='SalePrice ~ GrLivArea * C(OverallQual)', data=data_train)
mod_int_cat = mod_int_cat.fit()
print(mod_int_cat.summary2())

### Log-transformation of the response

House prices are right-skewed, and the spread of the residuals grows with the price (heteroscedasticity). ISLP 3.3.3 (Fig. 3.11) presents a concave transformation of the response such as $\log(Y)$ as a remedy. Let's compare the residual plots of `mod_01` and a log model. With *np.log()* we can specify the transformation on the fly.

In [ ]:
mod_log = smf.ols(formula='np.log(SalePrice) ~ GrLivArea', data=data_train)
mod_log = mod_log.fit()
print(mod_log.summary())

Residuals vs. fitted values, first for `mod_01` (SalePrice ~ GrLivArea, i.e., the same model without the log-transformation), then for `mod_log`. A funnel shape indicates heteroscedasticity.

In [ ]:
sns.scatterplot(x=mod_01.fittedvalues, y=mod_01.resid, alpha=0.3)
plt.axhline(0, color='red')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('SalePrice ~ GrLivArea')
plt.show()

In [ ]:
sns.scatterplot(x=mod_log.fittedvalues, y=mod_log.resid, alpha=0.3)
plt.axhline(0, color='red')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('log(SalePrice) ~ GrLivArea')
plt.show()

Note that $R^2$ is **not comparable** between the two models: `mod_log` explains variance in log(SalePrice), `mod_01` variance in SalePrice. To compare them on the same scale, we will back-transform the log predictions to dollars and compute the test RMSE below. In the log model, $e^{\beta_1}-1$ is the approximate relative change in price per extra sq ft.

In [ ]:
print('R² SalePrice ~ GrLivArea:     ', mod_01.rsquared)
print('R² log(SalePrice) ~ GrLivArea:', mod_log.rsquared)

### Polynomial transformation of the predictors

Fit a linear regression model with a second-order polynomial term for *GrLivArea*. This is still a linear model in the coefficients, just like eq. (3.36) in ISLP 3.3.2.

In [ ]:
mod_quad = smf.ols(formula='SalePrice ~ GrLivArea + I(GrLivArea**2)', data=data_train)
mod_quad = mod_quad.fit()
print(mod_quad.summary())

In formulas, `^` is not a power (in Python it's bitwise XOR), so wrap arithmetic in `I()`: `I(GrLivArea**2)` computes the square before it enters the model.

The summary warns about a large condition number: *GrLivArea* and its square are highly correlated (collinearity, ISLP 3.3.3). Centring *GrLivArea* before squaring would avoid this. Lab 3.6.6 uses `poly()`, which creates orthogonal polynomial terms: different coefficients, but the same fitted values.

## Make predictions

Use a trained model to make predictions for the test set.

In [ ]:
preds = mod_quad.predict(data_test)
preds

Calculate the test set RMSE, i.e., the square root of the test MSE. It is related to, but not the same as, the RSE (3.15) in the regression output, which is computed on the training data. For comparison, we also compute the training RMSE.

In [ ]:
rmse_test = root_mean_squared_error(data_test["SalePrice"], preds)
rmse_train = root_mean_squared_error(data_train["SalePrice"], mod_quad.fittedvalues)
print('Train RMSE:', rmse_train)
print('Test RMSE: ', rmse_test)

RMSE is in dollars: the typical size of our prediction error. Compare it with the price distribution from `describe()` above.

Now let's compare the simple model with the log model on the dollar scale. The log model predicts log(SalePrice), so we back-transform with `np.exp()` before computing the RMSE (strictly, this estimates the median rather than the mean price; we ignore this detail here).

In [ ]:
preds_01 = mod_01.predict(data_test)
preds_log = np.exp(mod_log.predict(data_test))

print('Test RMSE SalePrice ~ GrLivArea:     ', root_mean_squared_error(data_test["SalePrice"], preds_01))
print('Test RMSE log(SalePrice) ~ GrLivArea:', root_mean_squared_error(data_test["SalePrice"], preds_log))

Which model has the lower test RMSE in dollars? Note that nicer residuals do not automatically mean a lower dollar RMSE: the log model treats a 10% error on a cheap house like a 10% error on an expensive one, while RMSE weighs large dollar errors (on expensive houses) most.

## Your turn!

**Your task:** Fit a multiple linear regression model with more predictors (e.g., *GrLivArea*, *OverallQual*, *Neighborhood*, *LotArea*, *FullBath*) and compute its test RMSE. Does it beat `mod_quad`? Does adding predictors always lower the training RMSE? And the test RMSE?

In [ ]:
# YOUR CODE HERE